In [ ]:
%pip install engineeringkit


# Cauer's elliptic filter (1931)

Chebyshev filters ripple in one band and are monotonic in the other.
Wilhelm Cauer, working on telephone filters in Göttingen and at MIT,
spread the error over both bands with rational functions built from
Jacobi's elliptic functions, and placed zeros of transmission just past
the band edge. His book *Siebschaltungen* (1931) set out the theory. For a
given ripple and attenuation, the elliptic filter has the narrowest
transition of any filter of its order
(:func:`~engineeringkit.signal_processing.elliptic_lowpass`,
:func:`~engineeringkit.signal_processing.iir_order`). The date and
attribution follow the book's 1931 VDI-Verlag edition as cited in Cauer's
Swiss patent CH166337.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from engineeringkit.signal_processing import (
    butterworth_lowpass,
    chebyshev1_lowpass,
    db20,
    elliptic_lowpass,
    frequency_response,
    iir_order,
)
from engineeringkit.signal_processing.visualizers import plot_pole_zero

## Order needed for one specification

Pass 0-1 kHz within 0.5 dB, stop above 1.2 kHz by 60 dB, at 8 kHz.



In [ ]:
fs, f_pass, f_stop, ripple, attenuation = 8000.0, 1000.0, 1200.0, 0.5, 60.0
orders = {family: iir_order(f_pass, f_stop, ripple, attenuation, fs, family) for family in ("butterworth", "chebyshev", "elliptic")}
print(orders)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
designs = {
    f"Butterworth, n = {orders['butterworth']}": butterworth_lowpass(orders["butterworth"], f_pass * 1.06, fs, output="sos"),
    f"Chebyshev, n = {orders['chebyshev']}": chebyshev1_lowpass(orders["chebyshev"], ripple, f_pass, fs, output="sos"),
    f"elliptic, n = {orders['elliptic']}": elliptic_lowpass(orders["elliptic"], ripple, attenuation, f_pass, fs, output="sos"),
}
for name, sos in designs.items():
    f, H = frequency_response(sos, None, worN=8192, fs=fs)
    axes[0].plot(f, db20(np.maximum(np.abs(H), 1e-7)), label=name, lw=1)
axes[0].axvspan(f_pass, f_stop, color="0.9")
axes[0].set(xlabel="frequency [Hz]", ylabel="gain [dB]", ylim=(-100, 5), xlim=(0, 4000), title="Same specification")
axes[0].legend(fontsize=8)

## Ripple in both bands, zeros on the circle

The elliptic stopband touches -60 dB again and again; between the
touches the gain falls to zero at the transmission zeros, which lie on
the unit circle.



In [ ]:
n = orders["elliptic"]
b, a = elliptic_lowpass(n, ripple, attenuation, f_pass, fs)
plot_pole_zero(b, a, ax=axes[1])
axes[1].set_title(f"elliptic, order {n}")
fig.tight_layout()
zeros = np.roots(b)
assert np.allclose(np.abs(zeros), 1.0, atol=1e-6)

f, H = frequency_response(b, a, worN=np.linspace(0, f_pass, 2000), fs=fs)
gain_pass = db20(np.abs(H))
f, H = frequency_response(b, a, worN=np.linspace(f_stop, fs / 2, 4000), fs=fs)
gain_stop = db20(np.maximum(np.abs(H), 1e-12))
print(f"passband {gain_pass.min():.3f} to {gain_pass.max():.3f} dB, stopband peak {gain_stop.max():.1f} dB")
assert gain_pass.min() > -ripple - 1e-6 and gain_stop.max() < -attenuation + 1e-6
assert orders["elliptic"] < orders["chebyshev"] < orders["butterworth"]

plt.show()